# CoNLL-U Corpus Analysis
## EmotionalizTED Corpus — LREC 2026

This notebook analyses a directory of CoNLL-U files organised as:

```
conllu/conllu_file/<discipline>/*.conllu
```

**Sections**
1. Setup & shared utilities
2. Corpus preprocessing — per-file statistics & quality checks
3. Discipline-level statistics — tokens, sentences, averages
4. Corpus-wide plots — tokens, sentences, distributions
5. Adjective analysis — counts, ratios, top lemmas per discipline
6. Content word analysis — top lemmas, POS breakdown per discipline
7. Export — zip all outputs for download


In [13]:

import zipfile, os

zip_path = "/content/conllu_file.zip"  # adjust if you used a different name
extract_path = "/content/conllu"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Files extracted to:", extract_path)



✅ Files extracted to: /content/conllu


## 1. Setup & Shared Utilities

All shared constants, the folder-discovery helper, and the CoNLL-U line parser live here.
Every later section imports from this cell's namespace — run it first.


In [15]:
import os, re, csv, math, zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT      = "conllu/conllu_file"   # root that contains discipline sub-folders
CONLLU_DIR = "conllu"              # parent used for CSV outputs
OUT_STATS  = os.path.join(CONLLU_DIR, "csv_stats")
OUT_ADJ    = "plots_adjectives"
OUT_CONTENT = "plots_content_words"
OUT_POS    = "plots_pos"

for d in [OUT_STATS, OUT_ADJ, OUT_CONTENT, OUT_POS]:
    os.makedirs(d, exist_ok=True)

# ── POS sets ───────────────────────────────────────────────────────────────────
CONTENT_POS = {"NOUN", "PROPN", "ADJ", "VERB", "ADV"}
SENT_END_PUNCT = {".", "!", "?"}


# ── Folder discovery (handles root/discipline/ AND root/lang/discipline/) ──────
def find_discipline_folders(root):
    """Return list of (discipline_name, absolute_path) regardless of nesting depth."""
    result = []
    for entry in sorted(os.listdir(root)):
        entry_path = os.path.join(root, entry)
        if not os.path.isdir(entry_path):
            continue
        contents = os.listdir(entry_path)
        has_conllu = any(f.lower().endswith(".conllu") for f in contents)
        has_subdirs = any(os.path.isdir(os.path.join(entry_path, c)) for c in contents)

        if has_conllu:
            result.append((entry, entry_path))
        elif has_subdirs:
            for sub in sorted(os.listdir(entry_path)):
                sub_path = os.path.join(entry_path, sub)
                if os.path.isdir(sub_path):
                    result.append((sub, sub_path))
    return result


# ── CoNLL-U file parser ────────────────────────────────────────────────────────
def parse_conllu_file(path):
    """
    Read one .conllu file and return a dict with:
      tokens, sentences, adj_count, pos_counts (Counter), lemma_counts (Counter)
    Skips multiword tokens (1-2) and empty nodes (3.1).
    """
    tokens = 0
    sentences = 0
    adj_count = 0
    pos_counts = Counter()
    lemma_counts = Counter()

    with open(path, "r", encoding="utf-8") as fh:
        for raw in fh:
            line = raw.strip()
            if not line:
                continue
            if line.startswith("# sent_id"):
                sentences += 1
                continue
            if line.startswith("#"):
                continue
            cols = line.split("\t")
            if len(cols) < 4:
                continue
            tid = cols[0]
            if "-" in tid or "." in tid:   # skip MWT / empty nodes
                continue
            tokens += 1
            upos = cols[3]
            lemma = (cols[2] if cols[2] != "_" else cols[1]).strip().lower()
            pos_counts[upos] += 1
            if upos in CONTENT_POS and lemma:
                lemma_counts[lemma] += 1
            if upos == "ADJ":
                adj_count += 1

    return {
        "tokens": tokens,
        "sentences": sentences,
        "adj_count": adj_count,
        "pos_counts": pos_counts,
        "lemma_counts": lemma_counts,
    }


# ── Generic plot helpers ───────────────────────────────────────────────────────
def save_barh(words, counts, title, path, color="salmon"):
    """Horizontal bar chart, saved to path."""
    fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * len(words))))
    ax.barh(words[::-1], counts[::-1], color=color)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Frequency")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

def save_bar(labels, values, title, ylabel, path, color="steelblue", rot=45):
    """Vertical bar chart, saved to path."""
    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.1), 5))
    ax.bar(labels, values, color=color)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Discipline")
    ax.set_xticklabels(labels, rotation=rot, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()

print("✅ Setup complete. Shared utilities loaded.")

✅ Setup complete. Shared utilities loaded.


## 2. Corpus Preprocessing — Quality Checks

Three optional data-quality scans:
- **Stats header injection** — prepend token/sentence counts as CoNLL-U comments into every file.
- **Bracket-sentence finder** — detect sentences whose `# text` is a parenthesised non-speech token, e.g. `(Laughter)`.
- **Consecutive NaN finder** — detect runs of two or more consecutive tokens with `FORM == nan`.

Run these once to verify corpus integrity; outputs go to `conllu/`.


In [16]:
# ── 2a. Per-file stats header injection ───────────────────────────────────────

def inject_stats_headers(root=CONLLU_DIR, backup=True):
    """
    Prepend a small stats comment block to every .conllu file:
        # conllu_stats: begin
        # conllu_tokens: N
        # conllu_sentences: N
        # avg_tokens_per_sentence: X.XXXX
        # conllu_stats: end
    Existing blocks are replaced. Optionally writes a .bak backup.
    """
    def _remove_block(lines):
        s = e = None
        for i, l in enumerate(lines):
            if l.strip() == "# conllu_stats: begin":
                s = i
            elif l.strip() == "# conllu_stats: end" and s is not None:
                e = i; break
        return lines[:s] + lines[e+1:] if s is not None and e is not None else lines

    updated = []
    for dirpath, _, fnames in os.walk(root):
        for fn in sorted(fnames):
            if not fn.lower().endswith((".conllu", ".conll")):
                continue
            path = os.path.join(dirpath, fn)
            stats = parse_conllu_file(path)
            t, s = stats["tokens"], stats["sentences"]
            avg = t / s if s else 0.0
            block = [
                "# conllu_stats: begin",
                f"# conllu_tokens: {t}",
                f"# conllu_sentences: {s}",
                f"# avg_tokens_per_sentence: {avg:.4f}",
                "# conllu_stats: end", ""
            ]
            with open(path, "r", encoding="utf-8") as fh:
                lines = fh.read().splitlines()
            if backup:
                bak = path + ".bak"
                if not os.path.exists(bak):
                    open(bak, "w", encoding="utf-8").write("\n".join(lines))
            new_lines = block + _remove_block(lines)
            with open(path, "w", encoding="utf-8", newline="\n") as fh:
                fh.write("\n".join(new_lines))
            updated.append(path)
    print(f"Stats headers injected into {len(updated)} files.")

inject_stats_headers()

Stats headers injected into 110 files.


In [17]:
# ── 2b. Bracket-only sentence finder ─────────────────────────────────────────

BRACKET_RE = re.compile(r'^\s*\(.*\)\s*$', re.UNICODE)

def find_bracket_sentences(root=CONLLU_DIR):
    """Find sentences whose # text is a single parenthesised token, e.g. (Laughter)."""
    results = []
    for dirpath, _, fnames in os.walk(root):
        for fn in sorted(fnames):
            if not fn.lower().endswith((".conllu", ".conll")):
                continue
            path = os.path.join(dirpath, fn)
            lines = open(path, encoding="utf-8").read().splitlines()
            for idx, line in enumerate(lines):
                if not line.startswith("# text"):
                    continue
                parts = line.split("=", 1)
                if len(parts) < 2:
                    continue
                text_val = parts[1].strip()
                if BRACKET_RE.match(text_val):
                    sent_id = "UNKNOWN"
                    for j in list(range(max(0,idx-5), idx)) + list(range(idx+1, min(len(lines),idx+6))):
                        if lines[j].strip().startswith("# sent_id"):
                            sent_id = lines[j].split("=",1)[-1].strip(); break
                    results.append({
                        "domain": os.path.relpath(path, root).split(os.sep)[0],
                        "sent_id": sent_id, "file": path, "text": text_val
                    })

    df = pd.DataFrame(results)
    out = os.path.join(CONLLU_DIR, "bracket_sentences.csv")
    df.to_csv(out, index=False)
    print(f"Found {len(df)} bracket sentences → {out}")
    return df

df_brackets = find_bracket_sentences()

Found 5 bracket sentences → conllu/bracket_sentences.csv


In [18]:
# ── 2c. Consecutive NaN token finder ─────────────────────────────────────────

def find_consecutive_nan(root=CONLLU_DIR, min_run=2):
    """Find runs of ≥ min_run consecutive tokens where FORM == 'nan'."""
    results = []
    for dirpath, _, fnames in os.walk(root):
        for fn in sorted(fnames):
            if not fn.lower().endswith((".conllu", ".conll")):
                continue
            path = os.path.join(dirpath, fn)
            lines = open(path, encoding="utf-8").read().splitlines()
            block, sent_id = [], "UNKNOWN"
            for line in lines + [""]:
                if line.strip() == "":
                    if not block:
                        continue
                    run, run_start, tok_idx = 0, None, 0
                    for bl in block:
                        if bl.strip().startswith("# sent_id"):
                            sent_id = bl.split("=",1)[-1].strip()
                        if bl.strip().startswith("#") or len(bl.split("\t")) < 2:
                            continue
                        tok_idx += 1
                        form = bl.split("\t")[1].strip().lower()
                        if form == "nan":
                            run_start = run_start or tok_idx; run += 1
                        else:
                            if run >= min_run:
                                results.append({"file": path, "sent_id": sent_id,
                                                "start_token": run_start, "run_length": run})
                            run, run_start = 0, None
                    if run >= min_run:
                        results.append({"file": path, "sent_id": sent_id,
                                        "start_token": run_start, "run_length": run})
                    block = []
                else:
                    block.append(line)

    df = pd.DataFrame(results)
    out = os.path.join(CONLLU_DIR, "consecutive_nan.csv")
    df.to_csv(out, index=False)
    print(f"Found {len(df)} NaN runs → {out}")
    return df

df_nan = find_consecutive_nan()

Found 0 NaN runs → conllu/consecutive_nan.csv


## 3. Discipline-Level Statistics

Aggregate per-file stats (tokens, sentences, adjective count) across all disciplines.
Produces two CSVs:

| File | Contents |
|---|---|
| `csv_stats/file_stats.csv` | One row per .conllu file |
| `csv_stats/discipline_stats.csv` | Aggregated totals & averages per discipline |


In [19]:
# ── Collect per-file statistics ───────────────────────────────────────────────

file_rows = []
for disc, disc_path in find_discipline_folders(ROOT):
    for fn in sorted(os.listdir(disc_path)):
        if fn.lower().startswith("merged_") or not fn.lower().endswith(".conllu"):
            continue
        path = os.path.join(disc_path, fn)
        s = parse_conllu_file(path)
        file_rows.append({
            "discipline": disc,
            "file": fn,
            "path": path,
            "tokens": s["tokens"],
            "sentences": s["sentences"],
            "adjectives": s["adj_count"],
            "adj_per_token": s["adj_count"] / s["tokens"] if s["tokens"] else 0,
            "adj_per_sentence": s["adj_count"] / s["sentences"] if s["sentences"] else 0,
            "tokens_per_sentence": s["tokens"] / s["sentences"] if s["sentences"] else 0,
        })

df_files = pd.DataFrame(file_rows)
df_files.to_csv(os.path.join(OUT_STATS, "file_stats.csv"), index=False)
print(f"Per-file stats: {len(df_files)} files across {df_files['discipline'].nunique()} disciplines")
df_files.head()

Per-file stats: 110 files across 11 disciplines


,discipline,file,path,tokens,sentences,adjectives,adj_per_token,adj_per_sentence,tokens_per_sentence
0,Transcripts_Art,Art_10_de.conllu,conllu/conllu_file/de/Transcripts_Art/Art_10_d...,1443,112,64,0.044352,0.571429,12.883929
1,Transcripts_Art,Art_1_de.conllu,conllu/conllu_file/de/Transcripts_Art/Art_1_de...,4182,327,220,0.052606,0.672783,12.788991
2,Transcripts_Art,Art_2_de.conllu,conllu/conllu_file/de/Transcripts_Art/Art_2_de...,2243,154,140,0.062416,0.909091,14.564935
3,Transcripts_Art,Art_3_de.conllu,conllu/conllu_file/de/Transcripts_Art/Art_3_de...,2257,148,135,0.059814,0.912162,15.250000
4,Transcripts_Art,Art_4_de.conllu,conllu/conllu_file/de/Transcripts_Art/Art_4_de...,2233,142,139,0.062248,0.978873,15.725352


In [20]:
# ── Aggregate by discipline ───────────────────────────────────────────────────

df_disc = df_files.groupby("discipline").agg(
    num_files=("file", "count"),
    total_tokens=("tokens", "sum"),
    total_sentences=("sentences", "sum"),
    total_adjectives=("adjectives", "sum"),
).reset_index()

df_disc["avg_tokens_per_file"]     = df_disc["total_tokens"]     / df_disc["num_files"]
df_disc["avg_sentences_per_file"]  = df_disc["total_sentences"]  / df_disc["num_files"]
df_disc["avg_tokens_per_sentence"] = df_disc["total_tokens"]     / df_disc["total_sentences"]
df_disc["adj_per_token"]           = df_disc["total_adjectives"] / df_disc["total_tokens"]

df_disc = df_disc.sort_values("discipline").reset_index(drop=True)
df_disc.to_csv(os.path.join(OUT_STATS, "discipline_stats.csv"), index=False)
print("Discipline summary:")
df_disc

Discipline summary:


,discipline,num_files,total_tokens,total_sentences,total_adjectives,avg_tokens_per_file,avg_sentences_per_file,avg_tokens_per_sentence,adj_per_token
0,Transcripts_Art,10,25056,1861,1371,2505.6,186.1,13.463729,0.054717
1,Transcripts_Business,10,24239,1772,1375,2423.9,177.2,13.678894,0.056727
2,Transcripts_Education,10,26686,1835,1402,2668.6,183.5,14.542779,0.052537
3,Transcripts_Entertainment,10,27009,2076,1502,2700.9,207.6,13.010116,0.055611
4,Transcripts_History,10,26288,1929,1307,2628.8,192.9,13.627786,0.049719
5,Transcripts_Medicine,10,24114,1525,1596,2411.4,152.5,15.812459,0.066186
6,Transcripts_Philosophy,10,22753,1828,1220,2275.3,182.8,12.446937,0.053619
7,Transcripts_Politics and Law,10,23011,1683,1445,2301.1,168.3,13.672608,0.062796
8,Transcripts_Psychology,10,23597,1829,1349,2359.7,182.9,12.901586,0.057168
9,Transcripts_Science,10,23079,1573,1417,2307.9,157.3,14.671964,0.061398


## 4. Corpus-Wide Plots

Bar charts and distribution plots (boxplots, histograms, scatter) for the main
corpus metrics broken down by discipline. All plots are saved to `plots_pos/`.


In [21]:
# ── 4a. Bar charts: totals and averages per discipline ────────────────────────

disc_labels = df_disc["discipline"].tolist()

metrics = [
    ("total_tokens",         "Total Tokens",               "Total tokens"),
    ("total_sentences",      "Total Sentences",             "Total sentences"),
    ("avg_tokens_per_file",  "Avg Tokens per Text",         "Avg tokens per file"),
    ("avg_sentences_per_file","Avg Sentences per Text",     "Avg sentences per file"),
    ("avg_tokens_per_sentence","Avg Tokens per Sentence",   "Avg tokens/sentence"),
]

for col, title, ylabel in metrics:
    fname = f"{col}.png"
    save_bar(disc_labels, df_disc[col].tolist(), title, ylabel,
             os.path.join(OUT_POS, fname))
    print("Saved:", fname)

/tmp/ipykernel_12025/4269193743.py:111: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=rot, ha="right")


Saved: total_tokens.png


/tmp/ipykernel_12025/4269193743.py:111: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=rot, ha="right")


Saved: total_sentences.png


/tmp/ipykernel_12025/4269193743.py:111: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=rot, ha="right")


Saved: avg_tokens_per_file.png


/tmp/ipykernel_12025/4269193743.py:111: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=rot, ha="right")


Saved: avg_sentences_per_file.png


/tmp/ipykernel_12025/4269193743.py:111: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=rot, ha="right")


Saved: avg_tokens_per_sentence.png


In [22]:
# ── 4b. Boxplots — token & sentence distributions per discipline ──────────────

def boxplot_metric(df_files, metric_col, out_path):
    disciplines = sorted(df_files["discipline"].unique())
    fig, ax = plt.subplots(figsize=(max(10, len(disciplines) * 1.3), 6))
    data = [df_files[df_files["discipline"] == d][metric_col].dropna().values
            for d in disciplines]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5)
    for patch in bp["boxes"]:
        patch.set_facecolor("salmon")
    for med in bp["medians"]:
        med.set(color="black", linewidth=1.5)
    ax.set_xticks(range(1, len(disciplines) + 1))
    ax.set_xticklabels(disciplines, rotation=45, ha="right")
    ax.set_title(f"Distribution of '{metric_col}' per Discipline")
    ax.set_ylabel(metric_col)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print("Saved:", out_path)

boxplot_metric(df_files, "tokens",              os.path.join(OUT_POS, "boxplot_tokens.png"))
boxplot_metric(df_files, "sentences",           os.path.join(OUT_POS, "boxplot_sentences.png"))
boxplot_metric(df_files, "tokens_per_sentence", os.path.join(OUT_POS, "boxplot_tokens_per_sentence.png"))

Saved: plots_pos/boxplot_tokens.png
Saved: plots_pos/boxplot_sentences.png
Saved: plots_pos/boxplot_tokens_per_sentence.png


In [23]:
# ── 4c. Scatter — tokens vs sentences per file, coloured by discipline ─────────

disciplines = sorted(df_files["discipline"].unique())
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(9, 6))
for i, disc in enumerate(disciplines):
    sub = df_files[df_files["discipline"] == disc]
    ax.scatter(sub["sentences"], sub["tokens"], label=disc,
               color=colors[i % len(colors)], alpha=0.7)
ax.set_xlabel("Sentences per File")
ax.set_ylabel("Tokens per File")
ax.set_title("Tokens vs Sentences per File — by Discipline")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_POS, "scatter_tokens_vs_sentences.png"), dpi=150)
plt.close()
print("Saved: scatter_tokens_vs_sentences.png")

Saved: scatter_tokens_vs_sentences.png


In [24]:
# ── 4d. Histograms — token & sentence count distributions ─────────────────────

for metric in ["tokens", "sentences"]:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df_files[metric].dropna(), bins=25, color="steelblue", edgecolor="white")
    ax.set_xlabel(metric.capitalize())
    ax.set_ylabel("Number of files")
    ax.set_title(f"Histogram of {metric} per file (all disciplines)")
    plt.tight_layout()
    path = os.path.join(OUT_POS, f"hist_{metric}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print("Saved:", path)

Saved: plots_pos/hist_tokens.png
Saved: plots_pos/hist_sentences.png


## 5. Adjective Analysis

Two complementary views:

1. **Ratio plots** — total adjectives, adj/token ratio, avg adj per file per discipline.
2. **Top-N lemma plots** — most frequent adjective lemmas per discipline + one overall chart.

Outputs: `plots_adjectives/`


In [25]:
# ── 5a. Adjective ratio plots (from discipline summary) ───────────────────────

disc_labels = df_disc["discipline"].tolist()

adj_metrics = [
    ("total_adjectives", "Total Adjectives",            "Count"),
    ("adj_per_token",    "Adjective / Token Ratio",     "Ratio"),
]
for col, title, ylabel in adj_metrics:
    save_bar(disc_labels, df_disc[col].tolist(), title, ylabel,
             os.path.join(OUT_ADJ, f"{col}.png"), color="salmon")
    print("Saved:", col + ".png")

/tmp/ipykernel_12025/4269193743.py:111: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=rot, ha="right")


Saved: total_adjectives.png


/tmp/ipykernel_12025/4269193743.py:111: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(labels, rotation=rot, ha="right")


Saved: adj_per_token.png


In [26]:
# ── 5b. Top adjective lemmas — overall + per discipline ───────────────────────

overall_adj = Counter()
by_disc_adj = {}

for disc, disc_path in find_discipline_folders(ROOT):
    by_disc_adj.setdefault(disc, Counter())
    for fn in sorted(os.listdir(disc_path)):
        if fn.lower().startswith("merged_") or not fn.lower().endswith(".conllu"):
            continue
        s = parse_conllu_file(os.path.join(disc_path, fn))
        # Re-collect adj lemmas specifically
        path = os.path.join(disc_path, fn)
        with open(path, encoding="utf-8") as fh:
            for raw in fh:
                cols = raw.strip().split("\t")
                if len(cols) > 3 and cols[3] == "ADJ":
                    lemma = (cols[2] if cols[2] != "_" else cols[1]).strip().lower()
                    if lemma:
                        overall_adj[lemma] += 1
                        by_disc_adj[disc][lemma] += 1

TOP_N = 20

# Overall chart
words, counts = zip(*overall_adj.most_common(TOP_N)) if overall_adj else ([], [])
save_barh(list(words), list(counts),
          f"Top {TOP_N} Adjectives — Overall",
          os.path.join(OUT_ADJ, "top_adjectives_overall.png"), color="purple")
pd.DataFrame({"adjective": words, "count": counts}).to_csv(
    os.path.join(OUT_ADJ, "top_adjectives_overall.csv"), index=False)
print("Saved: top_adjectives_overall")

# Per-discipline charts + combined CSV
disc_frames = []
for disc, counter in sorted(by_disc_adj.items()):
    if not counter:
        continue
    top = counter.most_common(TOP_N)
    words, counts = zip(*top)
    save_barh(list(words), list(counts),
              f"Top {TOP_N} Adjectives — {disc}",
              os.path.join(OUT_ADJ, f"top_adjectives_{disc}.png"))
    df_tmp = pd.DataFrame({"discipline": disc, "adjective": words, "count": counts})
    df_tmp.to_csv(os.path.join(OUT_ADJ, f"top_adjectives_{disc}.csv"), index=False)
    disc_frames.append(df_tmp)
    print(f"Saved: top_adjectives_{disc}")

pd.concat(disc_frames, ignore_index=True).to_csv(
    os.path.join(OUT_ADJ, "top_adjectives_combined.csv"), index=False)
print("Saved: top_adjectives_combined.csv")

Saved: top_adjectives_overall
Saved: top_adjectives_Transcripts_Art
Saved: top_adjectives_Transcripts_Business
Saved: top_adjectives_Transcripts_Education
Saved: top_adjectives_Transcripts_Entertainment
Saved: top_adjectives_Transcripts_History
Saved: top_adjectives_Transcripts_Medicine
Saved: top_adjectives_Transcripts_Philosophy
Saved: top_adjectives_Transcripts_Politics and Law
Saved: top_adjectives_Transcripts_Psychology
Saved: top_adjectives_Transcripts_Science
Saved: top_adjectives_Transcripts_Technology
Saved: top_adjectives_combined.csv


## 6. Content Word Analysis

Content words = NOUN, PROPN, ADJ, VERB, ADV.

1. **Top-N lemma plots** — most frequent content word lemmas overall and per discipline.
2. **POS proportion plots** — how the five POS categories are distributed overall and per discipline.

Outputs: `plots_content_words/`


In [27]:
# ── 6a. Collect content word lemma counts ────────────────────────────────────

overall_cw  = Counter()
by_disc_cw  = {}
overall_pos = Counter()
by_disc_pos = {}

for disc, disc_path in find_discipline_folders(ROOT):
    by_disc_cw.setdefault(disc, Counter())
    by_disc_pos.setdefault(disc, Counter())
    for fn in sorted(os.listdir(disc_path)):
        if fn.lower().startswith("merged_") or not fn.lower().endswith(".conllu"):
            continue
        s = parse_conllu_file(os.path.join(disc_path, fn))
        for lemma, cnt in s["lemma_counts"].items():
            overall_cw[lemma]      += cnt
            by_disc_cw[disc][lemma] += cnt
        for pos, cnt in s["pos_counts"].items():
            if pos in CONTENT_POS:
                overall_pos[pos]       += cnt
                by_disc_pos[disc][pos]  += cnt

total_content = sum(overall_pos.values())
print(f"Total content word tokens: {total_content:,}")
print("Overall POS breakdown:", dict(overall_pos.most_common()))

Total content word tokens: 110,433
Overall POS breakdown: {'NOUN': 34712, 'ADV': 28808, 'VERB': 27239, 'ADJ': 15381, 'PROPN': 4293}


In [28]:
# ── 6b. Top content word lemma plots ─────────────────────────────────────────

TOP_N = 20

# Overall
words, counts = zip(*overall_cw.most_common(TOP_N)) if overall_cw else ([], [])
save_barh(list(words), list(counts),
          f"Top {TOP_N} Content Words — Overall",
          os.path.join(OUT_CONTENT, "top_content_words_overall.png"), color="purple")
pd.DataFrame({"word": words, "count": counts}).to_csv(
    os.path.join(OUT_CONTENT, "top_content_words_overall.csv"), index=False)

# Per discipline
disc_frames = []
for disc, counter in sorted(by_disc_cw.items()):
    if not counter:
        continue
    top = counter.most_common(TOP_N)
    words, counts = zip(*top)
    save_barh(list(words), list(counts),
              f"Top {TOP_N} Content Words — {disc}",
              os.path.join(OUT_CONTENT, f"top_content_words_{disc}.png"))
    df_tmp = pd.DataFrame({"discipline": disc, "word": words, "count": counts})
    df_tmp.to_csv(os.path.join(OUT_CONTENT, f"top_content_words_{disc}.csv"), index=False)
    disc_frames.append(df_tmp)
    print(f"Saved: top_content_words_{disc}")

pd.concat(disc_frames, ignore_index=True).to_csv(
    os.path.join(OUT_CONTENT, "top_content_words_combined.csv"), index=False)
print("Saved: top_content_words_combined.csv")

Saved: top_content_words_Transcripts_Art
Saved: top_content_words_Transcripts_Business
Saved: top_content_words_Transcripts_Education
Saved: top_content_words_Transcripts_Entertainment
Saved: top_content_words_Transcripts_History
Saved: top_content_words_Transcripts_Medicine
Saved: top_content_words_Transcripts_Philosophy
Saved: top_content_words_Transcripts_Politics and Law
Saved: top_content_words_Transcripts_Psychology
Saved: top_content_words_Transcripts_Science
Saved: top_content_words_Transcripts_Technology
Saved: top_content_words_combined.csv


In [29]:
# ── 6c. POS proportion — overall bar chart ───────────────────────────────────

POS_tags = sorted(CONTENT_POS)
pos_props = [overall_pos.get(p, 0) / total_content for p in POS_tags]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar([p.capitalize() for p in POS_tags], pos_props, color="mediumseagreen")
for bar, v in zip(bars, pos_props):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.005,
            f"{v:.1%}", ha="center", fontsize=10)
ax.set_title("POS Proportions among Content Words — Overall")
ax.set_ylabel("Proportion")
ax.set_xlabel("Part of Speech")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUT_CONTENT, "pos_proportion_overall.png"), dpi=150)
plt.close()
print("Saved: pos_proportion_overall.png")

Saved: pos_proportion_overall.png


In [30]:
# ── 6d. POS proportion — grouped bar chart by discipline ──────────────────────

domains = sorted(by_disc_pos.keys())
x = np.arange(len(domains))
width = 0.15

fig, ax = plt.subplots(figsize=(max(8, len(domains)*1.2), 6))
for i, pos in enumerate(POS_tags):
    y = []
    for d in domains:
        total = sum(by_disc_pos[d].values())
        y.append(by_disc_pos[d].get(pos, 0) / total if total else 0)
    bars = ax.bar(x + i*width, y, width, label=pos.capitalize())
    for bar, val in zip(bars, y):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
                f"{val:.0%}", ha="center", fontsize=7)

ax.set_xticks(x + width * (len(POS_tags)/2 - 0.5))
ax.set_xticklabels([d.capitalize() for d in domains], rotation=45, ha="right")
ax.set_ylabel("Proportion")
ax.set_title("POS Proportions among Content Words — by Discipline")
ax.legend(title="POS", bbox_to_anchor=(1.05, 1), loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUT_CONTENT, "pos_proportion_by_discipline.png"), dpi=150)
plt.close()
print("Saved: pos_proportion_by_discipline.png")

Saved: pos_proportion_by_discipline.png


In [31]:
# ── 6e. POS proportion CSV ────────────────────────────────────────────────────

pos_rows = []
for d in domains:
    total = sum(by_disc_pos[d].values())
    for pos in POS_tags:
        cnt = by_disc_pos[d].get(pos, 0)
        pos_rows.append({"discipline": d, "POS": pos,
                         "count": cnt,
                         "proportion": cnt/total if total else 0})

df_pos = pd.DataFrame(pos_rows)
df_pos.to_csv(os.path.join(OUT_CONTENT, "pos_proportions_by_discipline.csv"), index=False)
print("Saved: pos_proportions_by_discipline.csv")

Saved: pos_proportions_by_discipline.csv


## 7. Export — Download All Outputs

Zip up all CSV and plot folders into a single archive and download it.


In [32]:
import zipfile, os

ZIP_PATH = "corpus_analysis_outputs.zip"

folders_to_zip = [OUT_STATS, OUT_ADJ, OUT_CONTENT, OUT_POS]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in folders_to_zip:
        for dirpath, _, fnames in os.walk(folder):
            for fn in fnames:
                full = os.path.join(dirpath, fn)
                arcname = os.path.relpath(full)   # keeps folder structure
                zf.write(full, arcname)

print(f"✅ Archive created: {ZIP_PATH}")
print(f"   Size: {os.path.getsize(ZIP_PATH) / 1024:.1f} KB")

✅ Archive created: corpus_analysis_outputs.zip
   Size: 1981.8 KB


In [33]:
# ── Download (Google Colab only) ──────────────────────────────────────────────
try:
    from google.colab import files
    files.download(ZIP_PATH)
    print("Download started.")
except ImportError:
    print(f"Not running in Colab. Find your archive at: {ZIP_PATH}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
